# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-course\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-12
8,9,송지민,M,69,서울,2025-12-27
54,55,윤영철,M,69,광주,2023-10-08
78,79,이서준,M,69,부산,2023-09-21
136,137,김선영,M,68,울산,2025-02-05
45,46,김서현,M,67,대전,2026-07-21
132,133,김성진,M,67,성남,2026-08-06
123,124,김시우,M,67,대구,2024-12-02
11,12,김정남,F,66,대전,2024-12-08
57,58,김성훈,F,66,성남,2025-08-22


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


In [11]:
print(category_sales["total_sales"].sum())
print(completed_sales_items["line_total"].sum())

148990000
148990000


## 11. 월별 매출


In [12]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,10254000,15,683600.0
8,2026-05,16831000,19,885842.0
9,2026-06,12994000,17,764353.0


In [13]:
print(monthly_summary["total_sales"].sum())
print(completed_order_sales["line_total"].sum())

148990000
148990000


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [14]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


In [22]:
print(customer_sales["total_sales"].sum())
print(completed_order_sales["line_total"].sum())

148990000
148990000


## 13. 결과 저장하기


In [15]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


In [23]:
check_category = pd.read_csv(
    REPORT_DIR / "ch04_category_sales.csv"
)

print(check_category.shape)
print(check_category.columns.tolist())

(7, 4)
['category', 'total_quantity', 'total_sales', 'sales_ratio']


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```
현재 데이터에서는 바로 실행이 어렵습니다. order_items에는 원래 'category'가 없기 때문입니다. merge를 활용해 products 데이터에서 병합해야합니다.
그리고 주문 상태에서 completed만 되어 있는 것을 봐야하기 때문에 orders와 병합하여 order_status == "completed" 조건을 적용해야 합니다.
orders.order_id의 중복 여부 확인 
- orders_itemsorders를 order_id 기준으로 병합 
- 병합 시 validate="many_to_one"과 indicator=True를 사용 
- 병합 전후 행 수와 _merge 값을 확인하여 미매칭 데이터가 없는지 검증 
- order_status == "completed"인 주문만 선택 
- products.product_id 중복 여부를 확인 
- products를 product_id 기준으로 병합
- validate="many_to_one"과 indicator=True로 검증

In [26]:
#가이드를 따라 LLM으로 작성한 코드

"""
completed 주문 기준 월별 품목 금액 합계(total_sales)와 고유 주문 수(order_count) 집계

주의: total_sales는 order_items의 quantity × unit_price 단순 합계입니다.
할인, 쿠폰, 배송비, 세금(VAT), 부분 환불, 매출 인식 시점 등이 반영되지 않았으므로
회계상 순매출(net sales)로 간주하면 안 됩니다.
"""
import pandas as pd

# DATA_DIR은 Notebook에서 이미 정의된 프로젝트 데이터 경로(pathlib.Path)를 사용
ORDERS_PATH = DATA_DIR / "orders.csv"
ITEMS_PATH = DATA_DIR / "order_items.csv"

# ------------------------------------------------------------
# 1. 데이터 로드
# ------------------------------------------------------------
orders = pd.read_csv(ORDERS_PATH)
items = pd.read_csv(ITEMS_PATH)
print(f"[로드] orders: {len(orders):,}행 / order_items: {len(items):,}행")

# ------------------------------------------------------------
# 2. line_total = quantity × unit_price
# ------------------------------------------------------------
for col in ["quantity", "unit_price"]:
    items[col] = pd.to_numeric(items[col], errors="coerce")
    n_bad = items[col].isna().sum()
    if n_bad:
        print(f"[경고] order_items.{col} 숫자 변환 실패/결측: {n_bad}행")

items["line_total"] = items["quantity"] * items["unit_price"]

# ------------------------------------------------------------
# 3. orders.order_id 고유성 확인 (many-to-one 전제 조건)
# ------------------------------------------------------------
dup_mask = orders["order_id"].duplicated(keep=False)
n_dup = orders.loc[dup_mask, "order_id"].nunique()
print(f"[고유성] orders.order_id 중복 ID 수: {n_dup}")
if n_dup:
    print(orders.loc[dup_mask].sort_values("order_id"))
    raise ValueError("orders.order_id가 고유하지 않아 many-to-one 병합을 진행할 수 없습니다.")

# ------------------------------------------------------------
# 4. many-to-one merge + indicator, 병합 전후 행 수 확인
# ------------------------------------------------------------
rows_before = len(items)
merged = items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one",   # orders 쪽 키가 중복이면 MergeError 발생
    indicator=True,
)
rows_after = len(merged)
print(f"[병합] 병합 전 order_items: {rows_before:,}행 → 병합 후: {rows_after:,}행")
assert rows_before == rows_after, "left many-to-one 병합 후 행 수가 달라졌습니다."

# ------------------------------------------------------------
# 5. 미매칭 확인
# ------------------------------------------------------------
print("[매칭] _merge 분포:")
print(merged["_merge"].value_counts().to_string())

# (a) order_items에는 있으나 orders에 없는 order_id
unmatched_items = merged.loc[merged["_merge"] == "left_only"]
print(f"[미매칭] orders에 없는 order_items 행: {len(unmatched_items)}"
      f" (고유 order_id {unmatched_items['order_id'].nunique()}개)")

# (b) orders에는 있으나 품목이 없는 주문 (left merge로는 안 보이므로 별도 확인)
orders_without_items = orders.loc[~orders["order_id"].isin(items["order_id"])]
print(f"[미매칭] 품목이 없는 orders: {len(orders_without_items)}건")

# 매칭된 행만 사용
matched = merged.loc[merged["_merge"] == "both"].drop(columns="_merge").copy()

# ------------------------------------------------------------
# 6. 날짜 변환 실패 확인
# ------------------------------------------------------------
matched["order_date_parsed"] = pd.to_datetime(
    matched["order_date"], format="%Y-%m-%d", errors="coerce"
)
bad_date = matched["order_date_parsed"].isna()
print(f"[날짜] 변환 실패 행: {bad_date.sum()}"
      f" (고유 order_id {matched.loc[bad_date, 'order_id'].nunique()}개)")
if bad_date.any():
    print(matched.loc[bad_date, ["order_id", "order_date"]].drop_duplicates().head(20))

matched = matched.loc[~bad_date]

# ------------------------------------------------------------
# 7. completed만 필터링 (공백/대소문자 정규화 후 비교)
# ------------------------------------------------------------
matched["order_status_norm"] = matched["order_status"].astype(str).str.strip().str.lower()
print("[상태] order_status 분포(품목 행 기준):")
print(matched["order_status_norm"].value_counts(dropna=False).to_string())

completed = matched.loc[matched["order_status_norm"] == "completed"].copy()
print(f"[필터] completed 품목 행: {len(completed):,}"
      f" / 고유 주문: {completed['order_id'].nunique():,}")

# ------------------------------------------------------------
# 8. 월별 total_sales, 고유 order_count
# ------------------------------------------------------------
completed["order_month"] = completed["order_date_parsed"].dt.to_period("M")

monthly = (
    completed.groupby("order_month")
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),   # 품목 행 수가 아닌 고유 주문 수
    )
    .reset_index()
    .sort_values("order_month")
)

# 검증: 월별 합계 = completed 전체 합계
assert monthly["total_sales"].sum() == completed["line_total"].sum()
assert monthly["order_count"].sum() == completed["order_id"].nunique()  # 주문 1건 = 날짜 1개 전제

print("\n[결과] completed 주문 기준 월별 품목 금액 합계")
print("※ quantity × unit_price 단순 합계이며 회계상 순매출이 아닙니다.")
print(monthly.to_string(index=False))

monthly.to_csv("monthly_completed_sales.csv", index=False)

[로드] orders: 300행 / order_items: 764행
[고유성] orders.order_id 중복 ID 수: 0
[병합] 병합 전 order_items: 764행 → 병합 후: 764행
[매칭] _merge 분포:
_merge
both          764
left_only       0
right_only      0
[미매칭] orders에 없는 order_items 행: 0 (고유 order_id 0개)
[미매칭] 품목이 없는 orders: 0건
[날짜] 변환 실패 행: 0 (고유 order_id 0개)
[상태] order_status 분포(품목 행 기준):
order_status_norm
completed    474
cancelled    162
refunded     128
[필터] completed 품목 행: 474 / 고유 주문: 184

[결과] completed 주문 기준 월별 품목 금액 합계
※ quantity × unit_price 단순 합계이며 회계상 순매출이 아닙니다.
order_month  total_sales  order_count
    2025-09      2650000            5
    2025-10     16970000           19
    2025-11     11021000           13
    2025-12     21085000           23
    2026-01     14272000           16
    2026-02      4784000            7
    2026-03     18323000           25
    2026-04     10254000           15
    2026-05     16831000           19
    2026-06     12994000           17
    2026-07     10652000           15
    2026-08      5103000    

실제 컬럼/범위와 맞지 않은 부분: 주요 컬럼명과 completed 분석 범위는 실제 데이터와 일치하였다. 다만 CSV 경로가 orders.csv, order_items.csv로 작성되어 현재 Notebook의 DATA_DIR / "orders.csv" 방식과 맞지 않았다. 또한 숫자형 재변환과 주문 상태 문자열 정규화 등 현재 실습에서 필수적이지 않은 처리도 포함되어 있었다.

수정 후 사용하였을 때 기존 실습과 같은 결과들, 요청 사항에 맞는 내용들을 작성했음을 볼 수 있었다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [16]:
# 과제 코드를 아래에 작성하세요.


In [17]:
# 1. 40세 이상 고객을 추출합니다.
customers_over_40 = customers[
    customers["age"] >= 40
]

display(customers_over_40)
print("40세 이상 고객 수:", len(customers_over_40))

,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-08-20
3,4,조영호,F,55,울산,2026-06-21
6,7,이상현,F,53,인천,2025-02-19
8,9,송지민,M,69,서울,2025-12-27
9,10,유도현,F,62,울산,2024-12-14
...,...,...,...,...,...,...
140,141,김진호,M,40,광주,2023-11-17
142,143,김예은,F,43,대전,2024-03-11
144,145,조정호,M,59,성남,2024-03-11
145,146,김숙자,M,61,성남,2026-03-02


40세 이상 고객 수: 78


In [18]:
# 2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
cheap_products = products.sort_values(
    "price",
    ascending=True
).head(10)

display(cheap_products)

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


In [19]:
# 3. 결제수단별 주문 수와 비율을 계산합니다.
payment_count = orders["payment_method"].value_counts()

payment_ratio = (
    payment_count
    / payment_count.sum()
    * 100
).round(2)

print(payment_count)
print(payment_ratio)

payment_method
kakao_pay        79
naver_pay        77
bank_transfer    74
card             70
Name: count, dtype: int64
payment_method
kakao_pay        26.33
naver_pay        25.67
bank_transfer    24.67
card             23.33
Name: count, dtype: float64


In [20]:
# 4. 카테고리별 평균 상품 가격을 계산합니다.
category_price = (
    products
    .groupby("category", as_index=False)
    .agg(
        average_price=("price", "mean"),
    )
    .sort_values("average_price", ascending=False)
)

print(category_price)

  category  average_price
4       식품  137142.857143
1       뷰티  117687.500000
6       패션  115909.090909
3      스포츠  111578.947368
0       도서  106857.142857
5     전자기기  101588.235294
2     생활용품   96437.500000


In [21]:
# 5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
completed_order_amount = completed_order_sales["line_total"].sum()

amount_difference = all_order_amount - completed_order_amount

print("전체 주문 금액:", all_order_amount)
print("완료 주문 금액:", completed_order_amount)
print("금액 차이:", amount_difference)

전체 주문 금액: 255610000
완료 주문 금액: 148990000
금액 차이: 106620000


# 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.
orders와 order_items 데이터를 사용해서 completed 주문을 기준으로 고객별 구매 금액을 계산하는 코드를 작성해줘
[코드 작성시 참고 사항]
line_total = quanity x unit_price 계산할 것
orders에서 orders_id 중복 여부를 확인할 것
orders와 order_items 데이터를 order_id 기준으로 병합할 것
merge 시 validate="many_to_one", indicator=True 사용할 것
그다음 주문상태에서 "completed" 주분만 필터링할 것
customer_id별 total_sales 계산
주문 수는 order_id의 nunique()로 계산
고객별 결과를 total_sales가 높은 순서로 정렬

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


[Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: 예

2. 병합 검증
- orders.order_id 중복: 0
- orders merge 전 행 수: 764
- orders merge 후 행 수: 764
- orders merge 미매칭: 0
- products merge 전 행 수: 474
- products merge 후 행 수: 474
- products merge 미매칭: 0

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0

4. 교차 검증
- 완료 주문 상세 line_total 합계: 148,990,000
- category_sales 합계: 148,990,000
- monthly_summary 합계: 148,990,000
- customer_sales 합계: 148,990,000
- 합계 일치 여부: PASS

5. 저장 파일
- ch04_category_sales.csv: [존재]
- ch04_product_sales.csv: [존재]
- ch04_monthly_sales.csv: [존재]
- ch04_customer_sales.csv: [존재]

6. LLM 검증
- LLM이 빠뜨린 검증 항목:
핵심 검증 항목은 빠뜨리지 않았다. 오히려 기존 실습보다 orders에는 있지만 order_items에는 없는 주문 확인과 assert를 이용한 합계 검증이 추가되어 있었다.

7. 남은 질문
- 월별 total_sales 변화가 주문 건수 변화 때문인지, 주문 1건당 금액 변화 때문인지 구분해서 볼 수 있는지

# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 주 제출물은 실행 완료 Notebook `chapter04/chapter04.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 오현우
- GitHub ID: dhgusdn0312
- 작성일:2026.10.07
- 최종 제출 URL: https://github.com/dhgusdn0312/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
completed 주문을 기준으로 어떤 상품 카테고리의 금액이 가장 큰지 확인한다.
금액은 각 주문 상세 행의 quantity × unit_price로 계산한 line_total의 합계로 정의한다.

### 사용한 파일/컬럼
1. orders.csv
    order_id, order_status

2. order_items.csv
    order_id, product_id, quantity, unit_price, line_total

3. products.csv
    product_id, category


### 결과 관찰
실제 데이터의 주문 상태에는 completed, cancelled, refunded가 존재하였다. 이번 분석에서는 이 중 completed 주문만 분석 대상으로 선택하였다.
order_items에는 카테고리 정보가 없기 때문에 product_id를 기준으로 products와 병합해야 카테고리별 집계가 가능하였다.

### 나의 해석과 판단
왜 이 컬럼과 범위를 선택했는지 작성하세요.
: 카테고리별 completed 주문 금액을 계산하려면 상품의 카테고리와 주문 수량, 단가 정보가 필요하고, 완료된 주문만 선택하기 위해 주문 상태 정보가 필요하다. 그래서 category, quantity, unit_price, order_status와 병합에 필요한 key 컬럼을 사용하였다.
또한 이번 과제의 분석 범위가 completed 주문으로 정해져 있기 때문에 cancelled와 refunded 주문은 제외하고 order_status == "completed" 조건을 유지하였다.

### 업무·분석적 의미
분석 질문에 필요한 데이터와 컬럼을 먼저 정하면 불필요한 데이터를 사용하지 않고 분석 범위를 명확하게 유지할 수 있다.
또한 주문 정보와 상품 정보가 서로 다른 테이블에 저장되어 있어도 공통 key를 이용해 필요한 정보를 연결하면 하나의 분석 기준으로 집계할 수 있다.

### 한계와 추가 확인 사항
이번에 계산한 total_sales는 completed 주문의 quantity × unit_price 합계이며, 실제 회사의 순매출이나 이익을 의미하지 않는다.
실제 매출을 판단하려면 다른 데이터가 필요할 것 같다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: order_status == "completed"
- 정렬 기준: 상품 가격은 높은 순서와 낮은 순서로 확인하였고, 카테고리별·상품별 집계 결과는 total_sales가 높은 순서로 정렬하였다.
- 만든 파생 컬럼: line_total, order_month
- `line_total` 계산식: quantity × unit_price

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
order_status의 실제 값에는 completed, cancelled, refunded가 있었고, 이후 금액 집계에서는 completed 주문만 사용하였다.
line_total은 각 주문 상세 행의 quantity × unit_price로 계산하였다. 또한 order_date를 날짜형으로 변환한 뒤 월별 집계를 위해 order_month를 생성하였다.

### 나의 해석과 판단
필터 기준이 달라지면 집계 결과도 달라진다. 예를 들어 completed뿐만 아니라 cancelled와 refunded까지 포함하면 이번 과제에서 정의한 완료 주문 기준 금액보다 더 큰 값이 계산될 수 있다.
따라서 분석 질문에서 정한 주문 상태 범위를 집계 과정에서도 동일하게 유지하는 것이 중요하다고 판단하였다.

### 업무·분석적 의미
분석 목적에 맞는 조건으로 데이터를 먼저 필터링하면 이후 집계 결과가 어떤 범위를 의미하는지 명확하게 설명할 수 있다.
또한 line_total과 order_month 같은 파생 컬럼을 만들어 두면 상품별 금액 계산이나 월별 집계를 반복해서 수행하기 편하다.

### 한계와 추가 확인 사항
line_total은 수량과 단가만 이용한 계산값이기 때문에 할인, 배송비, 세금 등의 정보는 반영하지 않는다.
또한 이번 분석에서는 completed 주문만 사용했기 때문에 cancelled, refunded 주문이 어떤 특성을 가지는지는 별도로 분석하지 않았다.

## 3. merge 검증
- 병합한 데이터: order_items + orders, completed_order_sales + products
- 사용한 key: order_id, product_id
- `validate` 결과: many_to_one 조건에서 오류 없이 병합
- `indicator` 결과: 두 병합 모두 _merge 값이 both만 존재했고, left_only와 right_only는 0건
- 병합 전/후 행 수:
    order_items + orders: 764행 → 764행
    completed_order_sales + products: 474행 → 474행

![merge 검증](images/step03_merge.png)

### 결과 관찰
- orders.order_id의 중복 수는 0건이었다.
- order_items와 orders를 병합한 결과 행 수는 764행에서 764행으로 유지되었고, 미매칭 데이터는 없었다.
- completed_order_sales와 products를 병합한 결과도 474행에서 474행으로 유지되었고, 미매칭 상품은 없었다.

### 나의 해석과 판단
두 병합 모두 validate="many_to_one" 조건에서 오류가 발생하지 않았고, 병합 전후 행 수도 동일했다.
또한 indicator=True로 확인한 결과 모든 행이 both로 매칭되어 있었기 때문에 병합 과정에서 누락되거나 추가된 데이터 없이 정상적으로 연결되었다고 판단하였다.

### 업무·분석적 의미
병합 과정에서 key가 중복되거나 미매칭 데이터가 발생하면 행 수가 늘어나거나 일부 데이터가 누락될 수 있고, 그 결과 금액 합계가 실제보다 커지거나 작아질 수 있다.
따라서 집계 전에 key 중복, 병합 관계, 병합 전후 행 수, 미매칭 여부를 확인하는 과정이 필요하다.

### 한계와 추가 확인 사항
이번 병합에서는 key 중복과 미매칭이 발견되지 않았지만, 다른 데이터에서는 key의 중복이나 누락이 존재할 수 있으므로 같은 방식으로 확인해야 한다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의: order_status == "completed"인 주문만 사용
- 카테고리별 결과: total_quantity, total_sales를 집계하고 total_sales가 높은 순서로 정렬
- 상품별 결과: 상품별 total_quantity, total_sales를 집계하고 total_sales가 높은 순서로 정렬
- 월별 결과: 월별 total_sales와 고유 주문 수인 order_count를 계산
- 고객별 결과: 고객별 order_count와 total_sales를 계산하고 total_sales가 높은 순서로 정렬

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
completed 주문 상세 데이터는 474행이며, 고유 주문 수는 184건이었다.
카테고리별, 상품별, 월별, 고객별로 total_sales를 집계할 수 있었고, 월별 집계에서는 2025년 12월의 total_sales가 21,085,000으로 가장 높았다.
각 집계 결과는 total_sales 기준으로 정렬하여 상위 항목을 확인할 수 있었다.

### 나의 해석과 판단
같은 completed 주문 데이터를 기준으로 하더라도 카테고리, 상품, 월, 고객처럼 집계 기준을 바꾸면 서로 다른 관점에서 데이터를 볼 수 있었다.
특히 월별 결과를 통해 completed 주문 금액이 매월 동일하지 않고 시점에 따라 차이가 있다는 것을 확인할 수 있었다. 다만 현재 결과만으로 그 차이가 발생한 원인까지 판단할 수는 없다.

### 업무·분석적 의미
카테고리별 집계는 어떤 상품군의 completed 주문 금액이 큰지 확인하는 데 사용할 수 있고, 상품별 집계는 개별 상품의 금액 규모를 확인하는 데 사용할 수 있다.
월별 집계는 시기별 변화를 확인하는 데 사용할 수 있으며, 고객별 집계는 completed 주문 금액이 큰 고객을 확인하는 기초 자료로 활용할 수 있다.

### 한계와 추가 확인 사항
이번 total_sales는 completed 주문의 quantity × unit_price 합계이므로 회계상 순매출과 같다고 단정할 수 없다.
할인, 배송비, 세금, 실제 환불 금액 등의 정보가 포함되어 있지 않다. 또한 total_sales가 높다고 해서 판매 수량이 가장 많거나 수익성이 가장 높다고 판단할 수는 없다.

## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148,990,000
- 카테고리 합계: 148,990,000
- 월별 합계: 148,990,000
- 고객별 합계: 148,990,000
- 차이 여부: 차이 없음

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
같은 completed 주문 데이터를 기준으로 집계했다면 집계 기준이 카테고리, 월, 고객으로 달라져도 전체 total_sales 합계는 같아야 한다.
실제로 원본 completed line_total 합계와 카테고리별, 월별, 고객별 합계가 모두 148,990,000으로 동일하게 확인되었다. 따라서 각 집계 과정에서 completed 주문 범위가 동일하게 유지되었다고 판단하였다.
만약 합계가 다르게 나온다면 먼저 completed 필터가 동일하게 적용되었는지 확인하고, 그다음 병합 과정에서 행이 중복되거나 누락되지 않았는지 확인해야 한다. 월별 집계의 경우에는 날짜 변환 실패나 order_month 결측 여부도 확인할 필요가 있다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약: orders와 order_items를 사용해 completed 주문 기준 월별 total_sales와 고유 주문 수를 계산하도록 요청하였다. line_total 계산, order_id 고유성 확인, many-to-one 병합, indicator, 병합 전후 행 수, 미매칭, 날짜 변환 실패, completed 필터, order_id.nunique()를 포함하도록 조건을 제시하였다.
- 제안 코드 요약: line_total 생성 → orders.order_id 중복 확인 → order_items와 orders 병합 → 병합 결과 검증 → 날짜 변환 → completed 주문 필터 → 월별 total_sales, order_count 집계 순서로 작성되었다.
- 실제 컬럼/범위와 맞지 않은 부분: 주요 컬럼명과 completed 분석 범위는 실제 데이터와 일치하였다. 다만 처음 제안된 코드에서는 CSV 경로를 "orders.csv", "order_items.csv"로 직접 지정하여 현재 Notebook에서 사용하는 DATA_DIR 경로와 맞지 않았다.
- 수정한 내용: 실제 Notebook 환경에 맞게 DATA_DIR / "orders.csv", DATA_DIR / "order_items.csv" 경로를 사용하도록 수정하였다.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM이 작성한 코드는 실행에 필요한 핵심 검증 항목을 대부분 포함하고 있었고, 실제 데이터에서도 orders.order_id 중복 0건, 병합 전후 764행 유지, 미매칭 0건, 날짜 변환 실패 0건으로 확인되었다.
하지만 처음 제안된 코드는 현재 Notebook의 실제 파일 경로와 맞지 않았기 때문에 그대로 실행하기보다 현재 환경과 데이터 구조에 맞는지 확인하고 수정한 뒤 사용하는 것이 필요했다.
또한 코드가 오류 없이 실행되더라도 실제 컬럼명, 주문 상태 범위, 병합 관계, 집계 기준이 분석 질문과 일치하는지 확인해야 분석 결과를 신뢰할 수 있다고 판단하였다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1.completed 주문 기준 카테고리별 total_sales는 스포츠가 31,743,000으로 가장 높았고, 전체 completed 주문 금액의 21.31%를 차지하였다.
2.월별 total_sales는 2025년 12월이 21,085,000으로 가장 높았고, 월에 따라 completed 주문 금액에 차이가 있었다.

### 그 결과를 뒷받침하는 수치/표
스포츠: total_quantity 295, total_sales 31,743,000, sales_ratio 21.31%
전자기기: total_sales 26,400,000으로 두 번째로 높았다.
2025년 12월: total_sales 21,085,000, order_count 23건
2026년 3월: total_sales 18,323,000, order_count 25건

### 추가로 확인하고 싶은 질문
카테고리별 total_sales 차이가 판매 수량의 차이 때문인지 상품 가격의 차이 때문인지 확인해 보고 싶다.

### 현재 결과의 한계
이번 분석의 total_sales는 completed 주문의 quantity × unit_price 합계이므로 실제 회사의 회계상 순매출이나 이익과 동일하다고 볼 수 없다.
또한 카테고리별 total_sales가 높다고 해서 해당 카테고리가 가장 인기 있거나 수익성이 가장 높다고 판단할 수 없다. 판매 수량, 상품 단가, 고객 수, 원가 등의 정보를 함께 확인해야 더 정확한 해석이 가능하다.
월별 금액 차이도 확인할 수 있었지만, 현재 데이터만으로 그 변화가 발생한 원인을 단정할 수는 없다.

## 최종 제출 체크
- [/] 핵심 셀 Output이 남아 있습니다.
- [/] merge와 총합 검증 Evidence가 있습니다.
- [/] 결과 관찰과 해석이 구분되어 있습니다.
- [/] LLM 코드를 검증했습니다.
- [/] 개인정보/Secret이 없습니다.
- [/] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [/] 최종 Notebook 파일 URL을 제출합니다.